# 14 Feedback to planner and execution bridge

Promote rows that became canonical-ready after accepted suggestions back into planner-style outputs. Optionally merge them with the latest dry-run plan and build execution manifests.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


PROJECT_ROOT = C:\00_Developement\sch-file-organizer
OUTPUT_DIR = C:\00_Developement\sch-file-organizer\data\outputs


In [2]:
from src.feedback_to_execution import (
    PromotionConfig,
    build_promoted_plan,
    merge_existing_plan,
    promotion_summary,
)

try:
    from src.executor import build_execution_bundle, save_manifest_bundle, manifest_summary
    HAS_EXECUTOR = True
except Exception as exc:
    HAS_EXECUTOR = False
    EXECUTOR_IMPORT_ERROR = repr(exc)

HAS_EXECUTOR


True

In [3]:
def latest_output(prefix: str, suffix: str = '.parquet'):
    candidates = sorted(OUTPUT_DIR.glob(f'{prefix}_*{suffix}'), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None

FEEDBACK_RERUN_PATH = latest_output('canonical_feedback_rerun')
EXISTING_PLAN_PATH = latest_output('plan_dry_run')

print('FEEDBACK_RERUN_PATH =', FEEDBACK_RERUN_PATH)
print('EXISTING_PLAN_PATH =', EXISTING_PLAN_PATH)
if not FEEDBACK_RERUN_PATH:
    raise FileNotFoundError('No canonical_feedback_rerun_*.parquet found in data/outputs')


FEEDBACK_RERUN_PATH = C:\00_Developement\sch-file-organizer\data\outputs\canonical_feedback_rerun_20260307_145403.parquet
EXISTING_PLAN_PATH = C:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260307_094350.parquet


In [4]:
PROMOTE_ONLY_NEWLY_READY = True
MERGE_WITH_EXISTING_PLAN = True
BUILD_EXECUTION_MANIFESTS = True
DEFAULT_FULL_ROOT = None  # optional absolute root for planner_target_full_path

config = PromotionConfig(
    promote_only_newly_ready=PROMOTE_ONLY_NEWLY_READY,
    default_full_root=DEFAULT_FULL_ROOT,
)
config


PromotionConfig(promote_only_newly_ready=True, require_changed_or_keep_decision=True, keep_ready_same_path_rows=True, base_confidence_newly_ready=0.96, base_confidence_already_ready=0.9, default_full_root=None)

In [5]:
feedback_rerun = pd.read_parquet(FEEDBACK_RERUN_PATH)
existing_plan = pd.read_parquet(EXISTING_PLAN_PATH) if (MERGE_WITH_EXISTING_PLAN and EXISTING_PLAN_PATH) else pd.DataFrame()

promoted = build_promoted_plan(feedback_rerun, config=config)
merged_plan = merge_existing_plan(existing_plan, promoted)

print('promoted summary:', promotion_summary(promoted))
print('merged rows:', len(merged_plan))


promoted summary: {'rows': 4, 'move_rows': 0, 'keep_rows': 0, 'review_rows': 4, 'newly_ready_rows': 0}
merged rows: 4


In [6]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(promoted, ['relative_path', 'planner_action', 'planner_reason', 'planner_target_relative_path', 'accepted_fields', 'became_canonical_ready'])
display(promoted['planner_action'].fillna('missing').value_counts().rename_axis('planner_action').reset_index(name='count'))


,relative_path,planner_action,planner_reason,planner_target_relative_path,accepted_fields,became_canonical_ready
0,docs/PV15p473-01_PM_PER_environmental-approval...,manual_review,missing_required_fields,<NA>,phase;date;version;status,False
1,docs/Thumbs.db,manual_review,missing_required_fields,<NA>,,False
2,docs/a.txt,manual_review,missing_required_fields,<NA>,,False
3,docs/b.txt,manual_review,missing_required_fields,<NA>,,False


,planner_action,count
0,manual_review,4


In [7]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
promoted_csv = OUTPUT_DIR / f'plan_promoted_from_feedback_{stamp}.csv'
promoted_parquet = OUTPUT_DIR / f'plan_promoted_from_feedback_{stamp}.parquet'
merged_csv = OUTPUT_DIR / f'plan_with_feedback_{stamp}.csv'
merged_parquet = OUTPUT_DIR / f'plan_with_feedback_{stamp}.parquet'

promoted.to_csv(promoted_csv, index=False, encoding='utf-8-sig')
promoted.to_parquet(promoted_parquet, index=False)
merged_plan.to_csv(merged_csv, index=False, encoding='utf-8-sig')
merged_plan.to_parquet(merged_parquet, index=False)

print(promoted_csv)
print(promoted_parquet)
print(merged_csv)
print(merged_parquet)


C:\00_Developement\sch-file-organizer\data\outputs\plan_promoted_from_feedback_20260307_150615.csv
C:\00_Developement\sch-file-organizer\data\outputs\plan_promoted_from_feedback_20260307_150615.parquet
C:\00_Developement\sch-file-organizer\data\outputs\plan_with_feedback_20260307_150615.csv
C:\00_Developement\sch-file-organizer\data\outputs\plan_with_feedback_20260307_150615.parquet


In [8]:
if BUILD_EXECUTION_MANIFESTS and HAS_EXECUTOR:
    bundle = build_execution_bundle(merged_plan)
    print('manifest summary:', manifest_summary(bundle))
    saved = save_manifest_bundle(bundle, OUTPUT_DIR, f'feedback_bridge_{stamp}')
    saved
elif BUILD_EXECUTION_MANIFESTS and not HAS_EXECUTOR:
    print('Executor import unavailable:', EXECUTOR_IMPORT_ERROR)
else:
    print('BUILD_EXECUTION_MANIFESTS is False')


manifest summary: {'executable_rows': 0, 'keep_rows': 0, 'review_rows': 4, 'blocked_rows': 0, 'rollback_rows': 0}
